# Predictive-maintenance pipeline — Colab (T4 GPU)

Runs the full within-family comparison with the **continuous sliding window** protocol
(step-1 windows, every test window scored, 3 seeds, trivial baselines).

1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells. Results are written to `ml_pipeline/results/` and (optionally) copied to Google Drive.

Note: cloning pulls ~1.6 GB of Git LFS data (SCANIA CSVs) from GitHub on every fresh runtime.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU first."

## 1. Clone the repo (with LFS data)

In [ ]:
REPO_URL = "https://github.com/Nafis878/Continuous-.git"
WORKDIR = "/content/Continuous"

# Private repo? Use a GitHub token instead (do NOT save it in the notebook):
#   from getpass import getpass; tok = getpass("GitHub token: ")
#   REPO_URL = f"https://{tok}@github.com/Nafis878/Continuous-.git"

import os
!git lfs install
if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git lfs pull
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -q -r ml_pipeline/requirements.txt

## 3. Optional: Google Drive (N-CMAPSS data + saving results)

N-CMAPSS `.h5` files (~28 GB) are not in git. Put the ones you need in a Drive folder and set
`NCMAPSS_DIR`. Without them, N-CMAPSS is skipped and everything else still runs.

In [ ]:
USE_DRIVE = True
NCMAPSS_DRIVE_DIR = "/content/drive/MyDrive/N-CMAPSS"        # folder with N-CMAPSS_DS*.h5
RESULTS_DRIVE_DIR = "/content/drive/MyDrive/Continuous-results"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(NCMAPSS_DRIVE_DIR):
        os.environ["NCMAPSS_DIR"] = NCMAPSS_DRIVE_DIR
        print("N-CMAPSS dir:", NCMAPSS_DRIVE_DIR, os.listdir(NCMAPSS_DRIVE_DIR))
    else:
        print("No N-CMAPSS folder on Drive -> N-CMAPSS will be skipped.")

## 4. Run experiments

In [ ]:
SEEDS = "42 43 44"
RUN_ABLATION = False   # LSTM ablation study is the longest step; enable when needed

In [ ]:
!python ml_pipeline/run_family_comparison.py --seeds {SEEDS}

In [ ]:
!python ml_pipeline/run_condition_rul.py --seeds {SEEDS}

In [ ]:
!python ml_pipeline/run_scania_cost_tune.py --seeds {SEEDS}

In [ ]:
!python ml_pipeline/run_n_cmapss_multi.py --seeds {SEEDS}

In [ ]:
if RUN_ABLATION:
    !python ml_pipeline/run_ablation.py

## 5. Look at the results

In [ ]:
import pandas as pd
R = "ml_pipeline/results/family_comparison"
cols = ["dataset", "model", "rmse_mean", "rmse_std", "r2_mean", "last_rmse_mean", "rank"]
rul = pd.read_csv(f"{R}/rul_family_summary.csv")
display(rul[[c for c in cols if c in rul.columns]])
if os.path.exists(f"{R}/scania_family_summary.csv"):
    display(pd.read_csv(f"{R}/scania_family_summary.csv"))

## 6. Save results

In [ ]:
import shutil, time
stamp = time.strftime("%Y%m%d-%H%M%S")
archive = shutil.make_archive(f"/content/results-{stamp}", "zip", "ml_pipeline/results")
print("zip:", archive)
if USE_DRIVE:
    os.makedirs(RESULTS_DRIVE_DIR, exist_ok=True)
    shutil.copy(archive, RESULTS_DRIVE_DIR)
    print("copied to", RESULTS_DRIVE_DIR)
else:
    from google.colab import files
    files.download(archive)